In [1]:
# !pip install mlflow scikit-learn pandas --quiet

import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier 
from sklearn.metrics import accuracy_score, f1_score,log_loss, confusion_matrix, ConfusionMatrixDisplay



mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("mnist-classifier")
print("Tracking URI:", mlflow.get_tracking_uri())

2026/08/30 03:20:20 INFO mlflow.tracking.fluent: Experiment with name 'mnist-classifier' does not exist. Creating a new experiment.


Tracking URI: http://localhost:5000


In [2]:
from sklearn.datasets import fetch_openml
import numpy as np

mnist = fetch_openml('mnist_784', version=1, as_frame=False)

X, y = mnist.data, mnist.target

y = y.astype(np.uint8)

X_train, X_test = X[:60000], X[60000:]   # The first 60K data points are being used as train dataest and the rest 10K as test.
y_train, y_test = y[:60000], y[60000:]

In [7]:
def train_and_evaluate(hidden_layer_sizes=(100,), learning_rate_init=0.001):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layer_sizes,
        learning_rate_init=learning_rate_init,
        max_iter=20,
        random_state=42,
        early_stopping=True,
        validation_fraction=0.1
    )
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)

    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="macro")
    val_loss = log_loss(y_test, probs)
    return model, acc, f1, val_loss

In [8]:
import matplotlib.pyplot as plt

def train_and_log(hidden_layer_sizes=(100,), learning_rate=0.001, run_name=None):
    with mlflow.start_run(run_name=run_name):
    
        mlflow.log_param("hidden_layer_sizes", str(hidden_layer_sizes))
        mlflow.log_param("learning_rate", learning_rate)
        

        model, acc, f1, val_loss = train_and_evaluate(hidden_layer_sizes=hidden_layer_sizes, learning_rate_init=learning_rate)

        for epoch, (train_loss, acc) in enumerate(zip(model.loss_curve_, model.validation_scores_)):
            mlflow.log_metric("train_loss", train_loss, step=epoch)
            mlflow.log_metric("val_accuracy", acc, step=epoch)

        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_macro", f1)
        mlflow.log_metric("val_loss", val_loss)

        preds = model.predict(X_test)
        cm = confusion_matrix(y_test, preds)
        fig, ax = plt.subplots(figsize=(8, 8))
        disp = ConfusionMatrixDisplay(confusion_matrix=cm)
        disp.plot(ax=ax, cmap=plt.cm.Blues, values_format='d')
        plt.title(f"Confusion Matrix (Hidden: {hidden_layer_sizes}, LR: {learning_rate})")
        plt.savefig("confusion_matrix.png")
        plt.close()
        
        mlflow.log_artifact("confusion_matrix.png")

        mlflow.set_tag("team", "data-science")
        mlflow.sklearn.log_model(model, name="model",serialization_format="pickle")

        run_id = mlflow.active_run().info.run_id
        return run_id

baseline_run_id = train_and_log((100,), 0.001, run_name="mlp-baseline")

/home/rishikesh-vadlakonda/AIOps/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
2026/08/30 03:35:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run mlp-baseline at: http://localhost:5000/#/experiments/2/runs/812b07ce58ff4e7c98cd3f2d904e9087
🧪 View experiment at: http://localhost:5000/#/experiments/2


In [9]:
sweep_run_ids = []

grid = [
    ((64,), 0.001),
    ((64,), 0.01),
    ((128,), 0.001),
    ((128,), 0.01),
    ((256,), 0.001),
    ((256,), 0.01)
]

for hidden_layer_sizes, learning_rate in grid:
    rid = train_and_log(hidden_layer_sizes, learning_rate, run_name= f"mlp_{hidden_layer_sizes}_{learning_rate}")
    sweep_run_ids.append(rid)

print("Sweep run IDs:", sweep_run_ids)

/home/rishikesh-vadlakonda/AIOps/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
2026/08/30 03:38:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run mlp_(64,)_0.001 at: http://localhost:5000/#/experiments/2/runs/5f40128b0ab042d4986c3d0d0cac0c4d
🧪 View experiment at: http://localhost:5000/#/experiments/2


2026/08/30 03:38:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run mlp_(64,)_0.01 at: http://localhost:5000/#/experiments/2/runs/91d6745d460c478e890ee26e692e4768
🧪 View experiment at: http://localhost:5000/#/experiments/2


/home/rishikesh-vadlakonda/AIOps/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
2026/08/30 03:39:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

🏃 View run mlp_(128,)_0.01 at: http://localhost:5000/#/experiments/2/runs/88acbc4a9c524683bd454c6078c3ba65
🧪 View experiment at: http://localhost:5000/#/experiments/2


/home/rishikesh-vadlakonda/AIOps/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
2026/08/30 03:42:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run mlp_(256,)_0.001 at: http://localhost:5000/#/experiments/2/runs/2f1ccf8950b443d0928469565f73550d
🧪 View experiment at: http://localhost:5000/#/experiments/2


2026/08/30 03:43:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run mlp_(256,)_0.01 at: http://localhost:5000/#/experiments/2/runs/0326992403b04c27979d77859b8a2ab7
🧪 View experiment at: http://localhost:5000/#/experiments/2
Sweep run IDs: ['5f40128b0ab042d4986c3d0d0cac0c4d', '91d6745d460c478e890ee26e692e4768', '7a587dc6b14c422ca9af88df747f3381', '88acbc4a9c524683bd454c6078c3ba65', '2f1ccf8950b443d0928469565f73550d', '0326992403b04c27979d77859b8a2ab7']


In [10]:
runs_df = mlflow.search_runs(
    experiment_names=["mnist-classifier"],
    order_by=["metrics.val_loss ASC"],
)

display_cols = [c for c in runs_df.columns if c in (
    "run_id", "tags.mlflow.runName", "params.hidden_layer_sizes", "params.learning_rate", "metrics.val_loss", "metrics.f1_macro"
)]
print(runs_df[display_cols].head(10).to_string(index=False))

best_run = runs_df.iloc[0]
print(f"\nBest run: {best_run['run_id']}  (val_loss={best_run['metrics.val_loss']:.4f})")

                          run_id  metrics.val_loss  metrics.f1_macro params.hidden_layer_sizes params.learning_rate tags.mlflow.runName
5f40128b0ab042d4986c3d0d0cac0c4d          0.251967          0.948343                     (64,)                0.001     mlp_(64,)_0.001
812b07ce58ff4e7c98cd3f2d904e9087          0.277082          0.954728                    (100,)                0.001        mlp-baseline
90365bd3772d4438a2ad93c6a235cfc8          0.293091          0.952081                    (100,)                0.001        mlp-baseline
38b9b92a78f8465ba1b514c642b30b16          0.293091          0.952081                    (100,)                0.001        mlp-baseline
7a587dc6b14c422ca9af88df747f3381          0.369359          0.956141                    (128,)                0.001    mlp_(128,)_0.001
0326992403b04c27979d77859b8a2ab7          0.381027          0.921587                    (256,)                 0.01     mlp_(256,)_0.01
91d6745d460c478e890ee26e692e4768          0.3879